In [1]:
from utils import * 

In [2]:
import argparse
import re
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Iterable, List, Sequence, Tuple

# ---------------------------------------------------------------------
# Utilities
# ---------------------------------------------------------------------

def _clean_colname(name: str) -> str:
    s = str(name).strip().lower()
    s = s.replace("___", "_").replace(" ", "_").replace("-", "_").replace("/", "_")
    s = re.sub(r"[^0-9a-z_]+", "", s)
    s = re.sub(r"_{2,}", "_", s).strip("_")
    if re.match(r"^\d", s):
        s = f"x{s}"
    return s

def _make_unique(names: Sequence[str]) -> List[str]:
    seen, out = {}, []
    for n in names:
        if n not in seen:
            seen[n] = 0
            out.append(n)
        else:
            seen[n] += 1
            out.append(f"{n}_dup{seen[n]}")
    return out

def clean_columns(df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    old = list(df.columns)
    new = _make_unique([_clean_colname(c) for c in old])
    mapping = pd.DataFrame({"old_name": old, "new_name": new})
    df = df.copy()
    df.columns = new
    return df, mapping

def _as_numeric(s: pd.Series) -> pd.Series:
    return pd.to_numeric(s, errors="coerce")

def _reverse_score(x: pd.Series, min_val: float, max_val: float) -> pd.Series:
    return (min_val + max_val) - _as_numeric(x)

def _rowwise_sum(df: pd.DataFrame, cols: Sequence[str], min_valid_prop: float = 0.8) -> pd.Series:
    sub = df[cols].apply(_as_numeric)
    req = int(np.ceil(len(cols) * min_valid_prop))
    out = sub.sum(axis=1, skipna=True)
    out[sub.notna().sum(axis=1) < req] = np.nan
    return out

def _rowwise_mean(df: pd.DataFrame, cols: Sequence[str], min_valid_prop: float = 0.8) -> pd.Series:
    sub = df[cols].apply(_as_numeric)
    req = int(np.ceil(len(cols) * min_valid_prop))
    out = sub.mean(axis=1, skipna=True)
    out[sub.notna().sum(axis=1) < req] = np.nan
    return out

def _existing(df: pd.DataFrame, cols: Sequence[str]) -> List[str]:
    return [c for c in cols if c in df.columns]

def _dedupe(seq: Iterable[str]) -> List[str]:
    seen = set()
    out = []
    for x in seq:
        if x not in seen:
            out.append(x)
            seen.add(x)
    return out


# ---------------------------------------------------------------------
# Questionnaire scoring
# ---------------------------------------------------------------------

def score_erq(df: pd.DataFrame) -> Dict[str, pd.Series]:
    out = {}

    items = _existing(df, [f"erq_{i:04d}" for i in range(1, 11)])
    if len(items) == 10:
        sub = df[items].apply(_as_numeric)
        reapp = ["erq_0001","erq_0003","erq_0005","erq_0007","erq_0008","erq_0010"]
        supp  = ["erq_0002","erq_0004","erq_0006","erq_0009"]

        if "erq_reappraisal_score" not in df.columns:
            out["erq_reappraisal_score"] = _rowwise_sum(sub, reapp)

        if "erq_suppression_score" not in df.columns:
            out["erq_suppression_score"] = _rowwise_sum(sub, supp)

        if "erq_score" not in df.columns:
            out["erq_score"] = _rowwise_sum(sub, items)

    return out

def score_pss10(df: pd.DataFrame) -> Dict[str, pd.Series]:
    items = _existing(df, [f"pss10_{i}" for i in range(1, 11)])
    if len(items) != 10:
        return {}

    sub = df[items].apply(_as_numeric)
    for i in [4,5,7,8]:
        sub[f"pss10_{i}"] = _reverse_score(sub[f"pss10_{i}"], 0, 4)

    out = {}
    if "pss10_score" not in df.columns:
        out["pss10_score"] = _rowwise_sum(sub, items)

    out["pss10_perceived_helplessness_score"] = _rowwise_sum(
        sub, ["pss10_1","pss10_2","pss10_3","pss10_6","pss10_9","pss10_10"]
    )
    out["pss10_lack_self_efficacy_score"] = _rowwise_sum(
        sub, ["pss10_4","pss10_5","pss10_7","pss10_8"]
    )
    return out

def score_mpq(df: pd.DataFrame) -> Dict[str, pd.Series]:
    items = [c for c in df.columns if re.fullmatch(r"mpq\d+", c)]
    if len(items) < 50:
        return {}
    return {"mpq_score": _rowwise_mean(df, items)}

def score_sni(df: pd.DataFrame) -> Dict[str, pd.Series]:
    parts = [c for c in [
        "sni_network_size_score",
        "sni_network_diversity_score",
        "sni_embedded_networks_score"
    ] if c in df.columns]

    if len(parts) >= 2 and "sni_score" not in df.columns:
        sub = df[parts].apply(_as_numeric)
        out = sub.mean(axis=1, skipna=True)
        out[sub.notna().sum(axis=1) < 2] = np.nan
        return {"sni_score": out}

    return {}

def score_leq(df: pd.DataFrame) -> Dict[str, pd.Series]:
    leq_cols = [c for c in df.columns if c.startswith("leq_")]
    binary = []
    for c in leq_cols:
        if any(t in c for t in ["detail", "details", "specify"]):
            continue
        vals = set(_as_numeric(df[c]).dropna().unique())
        if vals and vals.issubset({0,1}):
            binary.append(c)

    if not binary:
        return {}

    count = _rowwise_sum(df, binary, min_valid_prop=0.5)
    return {
        "leq_event_count_score": count,
        "leq_score": count
    }

def score_stcq(df: pd.DataFrame) -> Dict[str, pd.Series]:
    out = {}

    def _block(cols, prefix):
        if not cols:
            return
        sub = df[cols].apply(_as_numeric)
        out[f"{prefix}_score"] = _rowwise_mean(sub, cols)
        out[f"{prefix}_sum_score"] = _rowwise_sum(sub, cols)

    screen = [c for c in df.columns if c.startswith("stcq_screen_") and not c.endswith("complete")]
    mri = [c for c in df.columns if c.startswith("stcq_mri_")]

    _block(screen, "stcq_screen")
    _block(mri, "stcq_mri")

    if "stcq_score" not in df.columns:
        if "stcq_screen_score" in out:
            out["stcq_score"] = out["stcq_screen_score"]
        elif "stcq_mri_score" in out:
            out["stcq_score"] = out["stcq_mri_score"]

    return out

# ---------------------------------------------------------------------
# Drug summaries
# ---------------------------------------------------------------------

def score_drug_summaries(df: pd.DataFrame) -> Dict[str, pd.Series]:
    out = {}

    ref_date = pd.to_datetime(df.get("initialconsent_date"), errors="coerce")

    def current_use(prefix):
        flags = []
        for c in [
            f"{prefix}_cur_number",
            f"{prefix}_cur_amount",
            f"{prefix}_cur_money",
            f"{prefix}_cur_duration_years",
            f"{prefix}_cur_dur_years"
        ]:
            if c in df.columns:
                flags.append((_as_numeric(df[c]).fillna(0) > 0))

        if f"{prefix}_cur_patt" in df.columns:
            flags.append(df[f"{prefix}_cur_patt"].notna())

        if flags:
            cur = flags[0]
            for f in flags[1:]:
                cur |= f
        elif f"{prefix}_last_use" in df.columns:
            last = pd.to_datetime(df[f"{prefix}_last_use"], errors="coerce")
            cur = (ref_date - last).dt.days <= 30
        else:
            cur = pd.Series(False, index=df.index)

        if f"{prefix}_hx" in df.columns:
            cur &= (_as_numeric(df[f"{prefix}_hx"]) == 1)

        return cur.astype(int)

    subs = ["alc","coc","thc","opi","amp","hall","inh","sedative","barb"]
    current = {}

    for s in subs:
        if any(c.startswith(s+"_") for c in df.columns):
            current[s] = current_use(s)
            out[f"drug_current_{s}"] = current[s]

    if "ftnd_smoke_status" in df.columns:
        current["nicotine"] = (_as_numeric(df["ftnd_smoke_status"]) == 1).astype(int)
        out["drug_current_nicotine"] = current["nicotine"]

    if current:
        curdf = pd.DataFrame(current)
        out["drug_n_current_types"] = curdf.sum(axis=1)
        illicit = [c for c in curdf.columns if c not in ["alc","nicotine"]]
        out["drug_n_current_illicit_types"] = curdf[illicit].sum(axis=1)

    hx_cols = [c for c in df.columns if c.endswith("_hx")]
    if hx_cols:
        hx = df[hx_cols].apply(_as_numeric)
        out["drug_n_lifetime_types"] = (hx == 1).sum(axis=1)

    def utox(prefix):
        cols = [c for c in df.columns if c.startswith(prefix+"utox_") and not c.endswith("neg")]
        if cols:
            return (df[cols].apply(_as_numeric) == 1).sum(axis=1)
        return None

    sc = utox("screen_")
    if sc is not None:
        out["drug_n_positive_utox_screen"] = sc

    mc = utox("mri_")
    if mc is not None:
        out["drug_n_positive_utox_mri"] = mc

    return out

# ---------------------------------------------------------------------
# Build pipeline
# ---------------------------------------------------------------------

@dataclass
class BuildOutputs:
    df_clean: pd.DataFrame
    df_full_clean: pd.DataFrame
    acronym_map: pd.DataFrame
    col_rename_map: pd.DataFrame

def build_clean_df(df_raw: pd.DataFrame) -> BuildOutputs:
    df, col_map = clean_columns(df_raw)

    # Rename existing totals to match convention
    rename = {
        "bdi_total": "bdi_score",
        "bdi_total_mri": "bdi_mri_score",
        "lsas_total_score": "lsas_score",
        "lsas_av_score": "lsas_avoidance_score",
        "stai_statescore": "stai_state_score",
        "stai_trait_total_score": "stai_trait_score",
        "ctq_physical_neglect_scale": "ctq_physical_neglect_score",
        "tas_total": "tas_score",
        "perceived_stress_scale_score": "pss10_score",
        "ders_total_score": "ders_score",
        "erq_cog_reappraisal_subscale": "erq_reappraisal_score",
        "erq_expressive_suppression_scubscale": "erq_suppression_score",
        "ecrs_total": "ecrs_score",
        "ecrs_anxiety_subscale": "ecrs_anxiety_score",
        "ecrs_avoidance_subscale": "ecrs_avoidance_score",
        "cssa_total": "cssa_score",
        "cssa_total_mri": "cssa_mri_score",
        "cq_total": "cq_score",
        "cq_total_mri": "cq_mri_score",
        "sds_total": "sds_score",
        "sni_network_size": "sni_network_size_score",
        "sni_network_diversity": "sni_network_diversity_score",
        "sni_embedded_networks": "sni_embedded_networks_score",
    }
    df = df.rename(columns={k:v for k,v in rename.items() if k in df.columns})

    # Compute scores
    for scorer in [
        score_erq,
        score_pss10,
        score_mpq,
        score_sni,
        score_leq,
        score_stcq,
        score_drug_summaries,
    ]:
        new = scorer(df)
        for k,v in new.items():
            if k not in df.columns:
                df[k] = v

    # STAI total
    if "stai_state_score" in df.columns and "stai_trait_score" in df.columns:
        df["stai_score"] = _as_numeric(df["stai_state_score"]) + _as_numeric(df["stai_trait_score"])

    # Final clean df = keep IDs + all *_score + drug summary vars
    id_cols = [c for c in ["sub_id","dx","age_years","sex"] if c in df.columns]
    score_cols = sorted([c for c in df.columns if c.endswith("_score")])
    drug_cols = sorted([c for c in df.columns if c.startswith("drug_")])

    df_clean = df[_dedupe(id_cols + score_cols + drug_cols)].copy()

    return BuildOutputs(
        df_clean=df_clean,
        df_full_clean=df.copy(),
        acronym_map=pd.DataFrame(),
        col_rename_map=col_map
    )


In [19]:
data_raw   = pd.read_excel('../data/questionnaire_data_raw.xlsx')
data_clean = build_clean_df(data_raw).df_clean

cols_to_delete = [c for c in data_clean.columns if c.endswith('_complete') or c.endswith('_timestamp')] + ['bdi_mri_score']
data_clean     = data_clean.drop(columns=cols_to_delete)

asi_cols   = [c for c in data_raw.columns if 'asi_' in c]
coc_cols   = [c for c in data_raw.columns if 'coc_' in c]
data_clean = data_clean.merge(data_raw[['sub_id'] + asi_cols + coc_cols], on='sub_id', how='left')
data_clean.to_excel('../data/questionnaire_data.xlsx', index=False)